[Guide Index](README.md)


# 05g Building on SHACL

An application that edits or displays shapes needs more than `validate()`: what kind of shape a node is, where a path leads, whether embedded SPARQL is sound, what a node expression gives, which properties a widget goes with. These are the functions `starlayer.shacl` offers for that, all importable from `starlayer.shacl` so that an application never reaches into a submodule. They answer questions about a shapes graph and a data graph; deciding what to show is the application's job.

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later sections reuse the shapes graph of section 1.

Graphs are `StarLayerGraph` and terms are pyoxigraph terms (see the [Graphs guide](02-graphs.ipynb)). Functions that take a graph accept a `StarLayerGraph`, a `StarLayerDataset` or a `pyoxigraph.Store` and raise `TypeError` for an rdflib graph.

In [1]:
import warnings

from starlayer.graph import Namespace, StarLayerGraph

warnings.simplefilter("ignore")  # the lexical-fidelity warning does not matter for these examples

EX = Namespace("http://example.org/")
SHAPES = """
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

    ex:PersonShape sh:targetClass ex:Person ;
      sh:property ex:NameShape, ex:FriendShape, ex:BossShape .
    ex:NameShape sh:path ex:name ; sh:minCount 1 ; sh:datatype xsd:string .
    ex:FriendShape sh:path [ sh:zeroOrMorePath ex:knows ] .
    ex:BossShape sh:path [ sh:inversePath ex:manages ] .
"""
shapes = StarLayerGraph()
shapes.parse(data=SHAPES, format="turtle")

DATA = """
    @prefix ex: <http://example.org/> .
    ex:ann a ex:Person ; ex:name "Ann" ; ex:knows ex:bo .
    ex:bo a ex:Person ; ex:name "Bo" ; ex:knows ex:cy .
    ex:cy a ex:Person ; ex:name "Cy" .
    ex:dee ex:manages ex:ann .
"""
data = StarLayerGraph()
data.parse(data=DATA, format="turtle")

<StarLayerGraph DEFAULT (9 triples)>

## 1. What a shapes graph is made of

### 1.1 The types a shapes graph entails

A shape need not say `a sh:NodeShape` or `a sh:PropertyShape`: SHACL 1.2 Core infers it from how the shape is used (a node with `sh:targetClass` is a node shape, a node with `sh:path` is a property shape). `infer_types(shapes_graph)` returns the `rdf:type` triples a shapes graph entails and does not already state, as a graph, by running the rules StarLayer ships (`starontology`, `shacl12-shacl-rules.ttl`).

In [2]:
from starlayer.shacl import infer_types

for s, _, o in sorted(infer_types(shapes).triples(), key=str):
    print(s.value.rsplit("/", 1)[-1], "is a", o.value.rsplit("#", 1)[-1])

BossShape is a PropertyShape
BossShape is a Shape
FriendShape is a PropertyShape
FriendShape is a Shape
NameShape is a PropertyShape
NameShape is a Shape
PersonShape is a NodeShape
PersonShape is a Shape


### 1.2 Which graph a shape belongs to

`shapes_graph_members(graph, shapes_graph)` returns the elements that belong to a `sh:ShapesGraph` by `rdfs:member`, `rdfs:isDefinedBy` or `owl:imports` ([4.e](05e-shacl-profiling.ipynb)), or `None` when the IRI is not a `sh:Graph` in the graph, so a mistyped IRI does not look like an empty module.

In [3]:
from starlayer.shacl import shapes_graph_members

packaged = StarLayerGraph()
packaged.parse(data="""
    @prefix ex: <http://example.org/> . @prefix sh: <http://www.w3.org/ns/shacl#> . @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Module a sh:ShapesGraph ; rdfs:member ex:Shared .
    ex:Own a sh:NodeShape ; rdfs:isDefinedBy ex:Module .
""", format="turtle")
print(sorted(m.value.rsplit("/", 1)[-1] for m in shapes_graph_members(packaged, EX.Module)))
print(shapes_graph_members(packaged, EX.NotAModule))

['Own', 'Shared']
None


### 1.3 Problems in embedded SPARQL

`meta_validate(shapes_graph)` also checks the queries a shapes graph holds (`sh:select`, `sh:ask`, `sh:construct`, `shui:searchQuery`): text that is not SPARQL, or that uses a prefix nothing declares (the queries of a shapes graph declare theirs with `sh:prefixes`; there is no ambient lookup). Each defect is a `sh:ValidationResult` in the `report_graph` of the `MetaShaclError`, with the node that holds the query as `sh:focusNode` and the property as `sh:resultPath`, so an editor can show the marks before the shapes are ever run.

In [4]:
from starlayer.graph.namespaces import RDF
from starlayer.shacl import SH, MetaShaclError, meta_validate

with_queries = StarLayerGraph()
with_queries.parse(data="""
    @prefix ex: <http://example.org/> . @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:Good a sh:NodeShape ; sh:sparql ex:GoodQuery .
    ex:GoodQuery sh:select "SELECT $this WHERE { $this <http://example.org/p> ?o }" .
    ex:Bad a sh:NodeShape ; sh:sparql ex:BadQuery .
    ex:BadQuery sh:select "SELECT $this WHERE { $this ex:p ?o }" .
    ex:Broken a sh:NodeShape ; sh:sparql ex:BrokenQuery .
    ex:BrokenQuery sh:select "SELECT $this WHERE { $this" .
""", format="turtle")
try:
    meta_validate(with_queries)
except MetaShaclError as error:
    report = error.report_graph
    for result in sorted(report.subjects(RDF.type, SH.ValidationResult), key=str):
        focus = next(iter(report.objects(result, SH.focusNode)))
        message = next(iter(report.objects(result, SH.resultMessage)))
        print(focus.value.rsplit("/", 1)[-1], "->", message.value[:100])

BadQuery -> sh:select value is not valid SPARQL: SPARQL syntax error at line 1, column 28: undeclared prefix 'ex
BrokenQuery -> sh:select value is not valid SPARQL: SPARQL syntax error at line 1, column 27: expected a predicate 


### 1.4 Parameter values of constraint components

`meta_validate()` also checks what a shapes graph gives for the parameters of its `sh:ConstraintComponent`s. SPARQL Extensions 4.2.1 says shapes that do not conform with the constraints declared for the parameters are ill-formed, and the engine does not check this when it validates: a query that compares text with a number silently finds nothing. The report names a shape whose value breaks its parameter's constraints, for example text given for an `xsd:integer` parameter. Only a parameter whose path is an IRI is checked.

## 2. Following a path

`path_values(data_graph, focus, path, shapes_graph=None)` follows a SHACL property path from a node through the data graph and returns the value nodes at its end as a `frozenset`. The path is written in the shapes graph (pass it as `shapes_graph=` when it is not the data graph). `is_inverse(shapes_graph, path)` says whether a path is an inverse path at its top level. A malformed path raises `ShapeError`.

In [5]:
from starlayer.graph import NamedNode
from starlayer.shacl import is_inverse, path_values

(friend_path,) = shapes.objects(EX.FriendShape, NamedNode("http://www.w3.org/ns/shacl#path"))
(boss_path,) = shapes.objects(EX.BossShape, NamedNode("http://www.w3.org/ns/shacl#path"))

reach = path_values(data, EX.ann, friend_path, shapes)
print("ann knows*:", sorted(n.value.rsplit("/", 1)[-1] for n in reach))
print("ann's boss (^manages):", [n.value.rsplit("/", 1)[-1] for n in path_values(data, EX.ann, boss_path, shapes)])
print("is_inverse:", is_inverse(shapes, boss_path), is_inverse(shapes, friend_path))

ann knows*: ['ann', 'bo', 'cy']
ann's boss (^manages): ['dee']
is_inverse: True False


## 3. Node expressions

`evaluate_expression(shapes_graph, data_graph, expression, focus=None, *, scope=None)` evaluates a node expression, a node of the shapes graph, for a focus node and returns the nodes it gives, in order; a failure that SHACL defines as "no result" raises `EvaluationFailure`. `scope` binds variables. `node_expression_functions()` and `node_expression_operators()` list the SPARQL functions an expression can call (`[ sparql:concat ( ... ) ]`) and the `shnex:` operators, as sorted IRIs, which is what a function picker offers ([4.a](05a-shacl-node-expressions.ipynb)).

In [6]:
from starlayer.shacl import evaluate_expression, node_expression_functions, node_expression_operators

expressions = StarLayerGraph()
expressions.parse(data="""
    @prefix ex: <http://example.org/> . @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    @prefix sparql: <http://www.w3.org/ns/sparql#> .
    ex:friends ex:expr [ shnex:pathValues ex:knows ] .
    ex:shout ex:expr [ sparql:ucase ( [ shnex:pathValues ex:name ] ) ] .
""", format="turtle")
(friends,) = expressions.objects(EX.friends, EX.expr)
(shout,) = expressions.objects(EX.shout, EX.expr)
print("friends of ann:", [n.value.rsplit("/", 1)[-1] for n in evaluate_expression(expressions, data, friends, EX.ann)])
print("name, upper-cased:", [n.value for n in evaluate_expression(expressions, data, shout, EX.bo)])
print(len(node_expression_functions()), "functions, e.g.", node_expression_functions()[:2])
print(len(node_expression_operators()), "operators, e.g.", [o.rsplit("#", 1)[-1] for o in node_expression_operators()[:4]])

friends of ann: ['bo']
name, upper-cased: ['BO']
78 functions, e.g. ['http://www.w3.org/ns/sparql#abs', 'http://www.w3.org/ns/sparql#bnode']
23 operators, e.g. ['arg', 'concat', 'conformsToShape', 'count']


## 4. Widgets and properties of a form

`select_widget(shape_node=, focus_node=, data_graph=, shapes_graph=, scoring_graph=, role=)` chooses the widget for one property shape; `select_widgets()` does it for every property of a node shape, and `explain_widgets()` says why ([4.d](05d-shacl-ui.ipynb)).

In [7]:
from starlayer.shacl import select_widget

ui = StarLayerGraph()
ui.parse(data="""
    @prefix ex: <http://example.org/> . @prefix sh: <http://www.w3.org/ns/shacl#> . @prefix shui: <http://www.w3.org/ns/shacl-ui/> .
    ex:PersonForm a sh:NodeShape ; sh:property ex:LabelProp, ex:AgeProp .
    ex:LabelProp sh:path ex:name ; shui:editor shui:TextFieldEditor .
    ex:AgeProp sh:path ex:age ; shui:editor shui:TextFieldEditor .
""", format="turtle")
print("widget:", select_widget(shape_node=EX.LabelProp, shapes_graph=ui, role="editor").value.rsplit("/", 1)[-1])

widget: TextFieldEditor


## 5. What a field can offer

`candidate_values(shapes_graph, property_shape, data_graph, *, focus=None, only_valid=False, subclass_of_in_shapes_graph=False)` answers what an editor asks for a field: what can a person pick? It returns a `Candidates` record with the `kind` of answer (`"list"` for a closed list from `sh:in` or `sh:hasValue`, `"instances"` from `sh:class` or `sh:rootClass`, `"open"` when there is nothing to list), the `values` in a stable order, whether the list is `closed` (the person may pick nothing else) and, when it is not, what can be `entry` typed (the `sh:datatype` or `sh:nodeKind`). The instances of a class include the instances of its subclasses, read from the same graph as `sh:class` is when validating, and several constraints on a field narrow the list: `sh:or` joins its alternatives and `sh:and` intersects them. The data graph supplies suggestions only: it is where the instances of a class are found, but whether a list is `closed` comes from the shape alone, and whether a value is valid is the verdict of the shapes, so a value the data does not back is still reported. Building shapes means checking that invalid data fails them. Labels are the application's job.

With `only_valid=True` and a `focus` node, each candidate is tried as the only value of the field for that node, on a copy of the data, and kept when the property shape still conforms. In the example below Ann is allergic to Tom, and the pet shape says `sh:disjoint ex:allergicTo`, so Tom is a cat and an animal but not a pet she can have. `StarShaclProcessor.candidate_values()` is the method form.

In [ ]:
from starlayer.graph import Namespace, StarLayerGraph
from starlayer.shacl import candidate_values

EX = Namespace("http://example.org/")
pet_shapes = StarLayerGraph()
pet_shapes.parse(data="""
    @prefix ex: <http://example.org/> . @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:PersonShape a sh:NodeShape ; sh:targetClass ex:Person ; sh:property ex:PetShape, ex:ColourShape .
    ex:PetShape sh:path ex:pet ; sh:class ex:Animal ; sh:disjoint ex:allergicTo .
    ex:ColourShape sh:path ex:colour ; sh:in ( ex:Red ex:Green ) .
""", format="turtle")
pets = StarLayerGraph()
pets.parse(data="""
    @prefix ex: <http://example.org/> . @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Dog rdfs:subClassOf ex:Animal . ex:Cat rdfs:subClassOf ex:Animal .
    ex:Rex a ex:Dog . ex:Tom a ex:Cat . ex:Happy a ex:Animal . ex:Fern a ex:Plant .
    ex:Ann a ex:Person ; ex:allergicTo ex:Tom .
""", format="turtle")

def names(found):
    return [term.value.rsplit("/", 1)[-1] for term in found.values]

every = candidate_values(pet_shapes, EX.PetShape, pets)
print("pet:", every.kind, names(every), "closed:", every.closed)
valid = candidate_values(pet_shapes, EX.PetShape, pets, focus=EX.Ann, only_valid=True)
print("pet that suit Ann:", names(valid))
colour = candidate_values(pet_shapes, EX.ColourShape, pets)
print("colour:", colour.kind, names(colour), "closed:", colour.closed)

pet: instances ['Happy', 'Rex', 'Tom'] closed: False
pet that suit Ann: ['Happy', 'Rex']
colour: list ['Green', 'Red'] closed: True


### 5.1 How a value is held, and whether one would be accepted

`value_form(shapes_graph, property_shape)` says how the value of a field is held: `"single"` (one term), `"list"` (an RDF list) or `"single-or-list"` (either one). It reads the property shape: a list-length or member constraint, `sh:class rdf:List` or the list pattern of the SHACL-for-SHACL shapes means a list, and an `sh:or` of a list with something that is not a list means either. The editor then knows whether to draw one input or a list.

`check_value(shapes_graph, property_shape, focus, value, data_graph)` says whether one value would be accepted for a field of a node, before it is saved. The value stands in for the field's current values on a copy of the data, and the node is checked against this property shape only. It returns a `ValueCheck` with `conforms` and the `results` as report records (section 6). It does not run the shapes-about-shapes check: a text that is not a valid regular expression is accepted for `sh:pattern` here, and found by `meta_validate()`. Both have method forms on `StarShaclProcessor`.

In [ ]:
from starlayer.shacl import check_value, value_form

field_shapes = StarLayerGraph()
field_shapes.parse(data="""
    @prefix ex: <http://example.org/> . @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:NickShape sh:path ex:nick ; sh:maxCount 1 .
    ex:LanguagesShape sh:path ex:languages ; sh:minListLength 1 .
    ex:TagsShape sh:path ex:tags ; sh:or ( [ sh:nodeKind sh:IRI ] [ sh:minListLength 1 ] ) .
""", format="turtle")
for shape in (EX.NickShape, EX.LanguagesShape, EX.TagsShape):
    print(shape.value.rsplit("/", 1)[-1], "->", value_form(field_shapes, shape))

for pick in (EX.Rex, EX.Tom):
    checked = check_value(pet_shapes, EX.PetShape, EX.Ann, pick, pets)
    print(pick.value.rsplit("/", 1)[-1], "as Ann's pet:", checked.conforms, [str(m.value) for r in checked.results for m in r.messages])

NickShape -> single
LanguagesShape -> list
TagsShape -> single-or-list
Rex as Ann's pet: True []
Tom as Ann's pet: False ['Value ex:Tom should not also be a value of ex:allergicTo']


## 6. Reading a report

A `ValidationReport` carries `conforms`, the `report_graph` and the `report_text`. `results` gives its results as records, `ReportResult`, in a stable order: most severe first, then by node and path. A record has the `focus`, the `path`, the `value`, the `component`, the `source_shape`, the `severity` (and `severity_name`), the `messages` and, for a nested shape, its `details`. Nested results are details of their parent, not results of their own. `for_node(node)`, `for_field(node, path)` and `node_level(node)` pick the results of a node, of one of its fields, or of the node as a whole, and `counts()` gives the number per severity. `deduplicated()` drops a result that is repeated exactly, which happens when two shapes that both target a node share a property shape.

`report_results(report_graph)` does the same for any report graph, such as the `report_graph` of a `MetaShaclError`. `diff_reports(old, new)` compares two reports and returns the results that are `added`, `fixed` and `unchanged`, which is what an editor shows after each edit. Results are matched by focus, path, value, component, source shape and severity, so the two validations have to use the same loaded shapes graph, and a blank focus node only matches by its label.

In [ ]:
from starlayer.shacl import diff_reports, validate

people_shapes = StarLayerGraph()
people_shapes.parse(data="""
    @prefix ex: <http://example.org/> . @prefix sh: <http://www.w3.org/ns/shacl#> . @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
    ex:PersonShape a sh:NodeShape ; sh:targetClass ex:Person ;
      sh:property [ sh:path ex:age ; sh:datatype xsd:integer ] ,
                  [ sh:path ex:nick ; sh:maxLength 3 ; sh:severity sh:Warning ] .
""", format="turtle")

def people(ann_age, bo_age):
    graph = StarLayerGraph()
    graph.parse(data=f'''
        @prefix ex: <http://example.org/> .
        ex:ann a ex:Person ; ex:age {ann_age} ; ex:nick "Annabel" .
        ex:bo a ex:Person ; ex:age {bo_age} .''', format="turtle")
    return graph

before = validate(people('"x"', "40"), people_shapes)
after = validate(people("41", '"y"'), people_shapes)

print("counts:", before.counts())
for result in before.results:
    print(result.severity_name, result.focus.value.rsplit("/", 1)[-1], result.path.value.rsplit("/", 1)[-1])
print("for ann's age:", len(before.for_field(EX.ann, EX.age)), "result")

change = diff_reports(before, after)
print("fixed:", [(r.focus.value.rsplit("/", 1)[-1], r.severity_name) for r in change.fixed])
print("added:", [(r.focus.value.rsplit("/", 1)[-1], r.severity_name) for r in change.added])
print("unchanged:", [(r.focus.value.rsplit("/", 1)[-1], r.severity_name) for r in change.unchanged])

counts: {'Violation': 1, 'Warning': 1, 'total': 2}
Violation ann age
Warning ann nick
for ann's age: 1 result
fixed: [('ann', 'Violation')]
added: [('bo', 'Violation')]
unchanged: [('ann', 'Warning')]


## 7. The form for a node

`form_model(shapes_graph, data_graph, node, *, role="editor", ...)` answers what an editor asks before it draws a form: which fields does this node have, and what is in them? It joins what the functions above give. The shapes that target the node are merged by path, so one field can come from several property shapes, and every contributing shape is kept in `property_shapes`. Fields are placed by the SHACL Core rule: ungrouped fields first by `sh:order`, then each `sh:group` by its own order, with `shui:defaultOrder` when a shape gives none. Each field carries its names and description, the widget for the role, the constraints that matter to an input (datatype, class, counts, length, pattern, ranges and `sh:in`), `value_form`, `required` and `editable`. `shapes=[shape]` shows one shape only, `empty` sets whether fields with no value are shown, and `placement=` replaces the order of the fields with the application's own.

Each value says where it came from. A value is `stored` in the data graph, `inferred` when it is found only in the `inferred_graph` that the caller passes, `computed` when `sh:values` gives it, or a `default` from `sh:defaultValue` that stands in for an empty field and is only a suggestion. Only a stored value in an editable field can be edited, so one field can show an editable value next to a read-only one, and `reason` says why a value is not editable. A field whose path is not a single property, such as a sequence, is included and read-only. The predicates of the node that no shape describes are in `others`.

Nothing here validates. Errors appear in a field only when the caller passes a `report`, candidate values only with `suggestions=True`, and `field.check(value)` asks `check_value` when a value is typed. The data graph supplies suggestions and values to show; the shapes and a report decide what is valid. `StarShaclProcessor.form_model()` is the method form.

In [ ]:
from starlayer.graph import Namespace, StarLayerGraph
from starlayer.shacl import form_model

EX = Namespace("http://example.org/")
person_shapes = StarLayerGraph()
person_shapes.parse(data="""
    @prefix ex: <http://example.org/> . @prefix sh: <http://www.w3.org/ns/shacl#> . @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
    ex:Contact a sh:PropertyGroup ; sh:name "Contact" ; sh:order 2 .
    ex:PersonShape a sh:NodeShape ; sh:targetClass ex:Person ;
      sh:property ex:NameShape, ex:MailShape .
    ex:NameShape sh:path ex:name ; sh:name "Name" ; sh:order 1 ; sh:minCount 1 ; sh:datatype xsd:string .
    ex:MailShape sh:path ex:mail ; sh:name "Mail" ; sh:group ex:Contact ; sh:datatype xsd:string ; sh:maxCount 1 .
""", format="turtle")
person = StarLayerGraph()
person.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:ann a ex:Person ; ex:name "Ann" ; ex:mail "ann@example.org" ; ex:age 41 .
""", format="turtle")
derived = StarLayerGraph()
derived.parse(data="""@prefix ex: <http://example.org/> . ex:ann ex:name "Annabel" .""", format="turtle")

def short(term):
    return term.value.replace("#", "/").rsplit("/", 1)[-1]


form = form_model(person_shapes, person, EX.ann, inferred_graph=derived)
for section in form.sections:
    print("section:", [label.value for label in section.label] or "(no group)")
    for field in section.fields:
        shown = [(value.term.value, value.source, value.editable) for value in field.values]
        print("  ", short(field.path), "required" if field.required else "optional", shown)
print("not described by a shape:", [short(field.path) for field in form.others])

section: (no group)
   name required [('Ann', 'stored', True), ('Annabel', 'inferred', False)]
section: ['Contact']
   mail optional [('ann@example.org', 'stored', True)]
not described by a shape: ['age', 'type']
